> **구분: 수업 핵심**

# 02. 이미지 질문과 근거 중심 프롬프트 — 60분

**질문 비교 10분 → grounding 10분 → prompt TODO 15분 → 실제 비교 20분 → 정리 5분** 입니다. 모델을 다시 훈련하지 않고 입력 문장의 계약을 바꿨을 때 답이 어떻게 달라지는지 관찰합니다.


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

같은 사진을 주어도 질문이 모호하면 관찰·추측·출력 형태가 섞일 수 있습니다. 이번 실습은 이미지와 모델 조건을 유지한 채 질문을 바꾸고, 사람이 먼저 적은 기대값으로 답의 근거를 평가하는 비교 실험입니다.

### 전체 과정에서 내 코드가 하는 일

```text
같은 이미지·모델
  ↓
기대 관찰 기록
  ↓
질문 TODO
  ↓
요청별 기록
  ↓
형식·의미·unknown 평가
  ↓
실패 건만 재시도
```

질문 필드와 불확실 처리 방식을 명시합니다. JSONL의 각 요청과 manifest의 공통 조건을 연결하고 성공한 응답은 재호출하지 않습니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 두 질문의 답이 다르면 두 번째가 더 좋은 질문인가요?

**설명:** 다르다는 사실만으로는 판단할 수 없습니다. 보이는 근거, 요구 형식, 가려진 정보의 보류를 각각 평가해야 합니다. 질문 외 모델·이미지·옵션을 함께 바꾸면 원인을 질문 하나에 돌릴 수 없습니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


<!-- notebook-restart:start -->
## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**커널 준비:** JupyterLab에서 **Python 3 (ipykernel)** 을 선택합니다. 새 커널에서는 이 노트북의 첫 Python 셀부터 실행합니다. 다른 단원의 모듈이 남아 있는 커널이라면 재시작한 뒤 준비 셀을 다시 실행합니다. 실행 경로는 `/workspace` 또는 현재 단원 폴더이고 Python은 `/opt/course/bin/python`입니다. 다른 단원과 실행 중인 커널을 공유하지 않습니다. 각 단원의 `common` 모듈 이름이 같아 이전 import가 남으면 다른 코드를 읽을 수 있습니다. `NameError`가 나면 필요한 초기화 셀부터 다시 실행합니다.

TODO는 지정된 `exercises/*.py`를 JupyterLab 편집기에서 수정·저장하고 **학습자 검증 셀** 을 다시 실행합니다. 이 노트북 안의 TODO는 해당 코드 셀을 직접 수정합니다. 미완성 상태의 `NotImplementedError`·검증 실패는 예상된 출발점입니다. 참고 정답 검증 통과는 학습자 파일의 통과와 별도입니다.

**2일차 VLM 연결:** 실제 질의에는 공용 Ollama가 필요합니다. 연결되지 않거나 모델이 없으면 로컬 Setup 담당자에게 서버·모델 저장소 확인을 요청합니다. 준비 여부는 **개인 컨테이너 JupyterLab Terminal** 에서 확인합니다.

```bash
cd /workspace
/opt/course/bin/python 05.VLM/scripts/check_environment.py
```

기본 API는 `http://127.0.0.1:11434`, 모델은 `gemma4:e2b-it-qat`이며 `vision` capability가 필요합니다. 주소·모델을 따로 배정받았다면 Python 초기화 셀에서 `os.environ['OLLAMA_HOST']`·`os.environ['VLM_MODEL']`을 설정합니다. 다른 Terminal에서 설정한 환경변수는 이미 열린 커널에 자동 적용되지 않습니다. 입력은 `Data/images/05.VLM/{road.png,dog.jpg}`, 선택 영상은 `Data/videos/05.VLM/ETS.mp4`, 출력은 `Data/outputs/05.VLM`입니다. 파일·모델이 없으면 배포 자산 경로를 확인합니다.

### 앞 실습의 위치와 이번에 다시 필요한 내용

[01_vlm_basics.ipynb의 1·3·4절](01_vlm_basics.ipynb)에 이미지 메시지와 응답 구조가 있습니다. 필요한 내용은 **PNG/JPEG bytes → Base64 문자열 → user message의 `images` 목록**, 질문은 `content`, 답은 `response['message']['content']`라는 점입니다. 이 노트북의 `client.chat_image(이미지경로, 질문, num_predict=120)`가 이미지 읽기·인코딩·메시지 구성을 모두 수행합니다. 01의 `message`·`encoded` 변수나 TODO 파일을 가져오지 않습니다. 비교할 두 질문과 기록 목록은 아래에서 새로 만듭니다.

<!-- notebook-restart:end -->

## 슬라이드와 함께 진행하기

노트북의 설명·예제·TODO를 먼저 진행한 뒤, 아래 복귀 지점에서 결과를 슬라이드로 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.

- **1–4절과 응답 비교표** → [2일차 슬라이드 60쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s055) · [배포 PDF 60쪽](../슬라이드/2일차.pdf#page=60)에서 66쪽까지 복습합니다.

수업 핵심 범위와 지정된 완료 검사를 진행하고 선택·자습 확장은 수업 후에 이어갑니다. 복습에는 방금 만든 결과를 사용하며, 기존 수업 시간 안에서 진행합니다.


## 정답 활용 안내

> 교재 버전: v1.0

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


> **구분: 수업 핵심**

수업 시작에 입력과 기대 결과를 예측하고, 종료 전 아래 두 완료 조건을 확인합니다.

## 이 노트북의 완료 확인과 단원 연결 질문

[작업대 상태 확인 사례](../공통사례.md) · [평가 워크북](../평가워크북.md)

- **이 노트북 완료:** 이 노트북에 표시된 핵심 TODO·검증·저장 결과를 확인합니다. 자습 노트북은 자습 범위만 확인합니다.
- **단원 전체 결과:** 같은 이미지의 요청·원문·검증 결과·지연을 저장합니다.
- **단원 마지막 연결 질문:** 공통 사례의 기대 label을 응답을 보기 전에 정합니다. 형식 통과와 업무 판단 정답을 따로 기록하고, 틀린 허용·불필요한 거부·판단 보류를 구분합니다.

각 단원의 마무리·비교 시간에 5분을 사용합니다. 별도 시간을 추가하지 않습니다. 마지막 5분은 저장·오류 복구에 우선 사용하고 심화 설명은 자습으로 넘깁니다. 결과와 짧은 이유를 `Data/outputs/course-review/<단원>/`의 개인 기록에 남깁니다. 자료가 부족하면 미측정으로 적으며 성공값을 만들어 넣지 않습니다.

USB 카메라 실습은 로컬 환경에서 한 번만 실행하고 두 수강생이 로컬 화면을 함께 봅니다. 역할을 교대하고 각자 결과를 기록합니다. [공동 실습 안내](../공동카메라실습.md)를 따릅니다. 녹화 경로는 재생·자습·장애 대응용 선택 경로입니다.


> **구분: 수업 핵심**

## 완료 조건

- 넓은 질문과 제한이 있는 질문의 차이를 설명합니다.
- `build_grounded_prompt`가 질문·근거 제한·`unknown`·field 순서를 포함합니다.
- 같은 이미지의 두 응답에서 이미지에 근거한 내용과 추측을 구분합니다.
- 모델 지연과 출력 처리량을 서로 다른 값으로 읽습니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import json, subprocess, sys, time
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / '05.VLM/common').is_dir())
COURSE = ROOT / '05.VLM'
if str(COURSE) not in sys.path: sys.path.insert(0, str(COURSE))
from common.paths import IMAGES, VIDEOS, OUTPUTS, LOGS, ensure_output_dirs
from common.assets import compact_image_info, encode_image
from common.client import OllamaVLMClient, timing_summary
from common.loading import load_op
from common.recording import create_prompt_comparison, record_prompt_comparison
ensure_output_dirs()
assert (IMAGES / 'road.png').is_file() and (IMAGES / 'dog.jpg').is_file(), '먼저 강사에게 Data/images/05.VLM 배포를 확인하세요.'
print('Python:', sys.executable)
print('input:', IMAGES)
print('output:', OUTPUTS)


> **구분: 수업 핵심**

## 1. 넓은 질문은 가능한 답의 범위도 넓습니다

`이 이미지를 설명해 줘`는 caption에는 쓸 수 있지만 어떤 정보가 꼭 필요한지 알 수 없습니다. 판단용 prompt에는 다음 네 가지를 분명히 씁니다.

1. **관찰 대상:** 객체, 색, 위치, 위험 요소 중 무엇을 볼지
2. **근거 범위:** 이미지에서 직접 보이는 내용만 사용할지
3. **불확실성:** 보이지 않거나 애매하면 `unknown`
4. **출력 항목:** 사람이 비교하거나 프로그램이 읽을 field와 순서

프롬프트를 자세히 쓴다고 틀린 답이 사라지지는 않습니다. 답을 검토할 기준을 구체화하는 효과가 있습니다.


In [ ]:
# 구분: 수업 핵심 셀
display(Image(filename=str(IMAGES / 'dog.jpg'), width=420))
broad = '이 이미지를 자세히 설명하세요.'
grounded_example = '''질문: 이미지에서 동물과 연결된 물체를 찾으세요.
이미지에서 직접 확인되는 근거만 사용하세요. 보이지 않거나 확실하지 않으면 unknown이라고 쓰세요.
반드시 다음 항목을 이 순서로 답하세요: animal, connected_object, visible_evidence'''
print('넓은 질문:\n', broad)
print('\n근거 중심 질문:\n', grounded_example)


> **구분: 수업 핵심**

## 2. TODO — 근거 중심 prompt 만들기

[exercises/prompts.py](exercises/prompts.py)의 `build_grounded_prompt(question, required_fields)`를 완성하세요. `required_fields`는 순서를 유지하는 문자열 list 또는 tuple입니다. 함수는 입력 원본을 변경하지 않아야 합니다.

빈 질문, 빈 field 목록, 문자열 하나를 목록처럼 준 경우, 중복 field와 빈 field는 `ValueError`로 거부합니다. field 이름을 그대로 포함해야 다음 단계에서 사람이 요구사항을 확인할 수 있습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `question='무엇이 보이나요?'`, `required_fields=['animal','visible_evidence']`를 받아 **prompt 문자열 하나** 를 반환합니다. dict나 모델 응답을 반환하지 않습니다.
**필수 내용:** 정리한 질문, 이미지에서 직접 보이는 근거만 사용하라는 지시, 불확실하면 `unknown`이라고 쓰라는 지시, field를 입력 순서대로 답하라는 지시가 모두 포함되어야 합니다.
**작성 흐름 힌트:** 질문 검사 → list/tuple 여부 검사 → 각 field의 앞뒤 공백을 제거한 새 목록 → 빈 값·비문자열·정리 후 중복 거부 → `', '.join(fields)`로 순서를 유지하여 문장 구성. 예를 들어 `['animal',' animal ']`은 중복입니다.
**설명할 지점:** “우리가 작성하는 것은 답이 아니라 답변 요구사항입니다. 형식을 잘 지켜도 사진 내용을 맞혔는지는 따로 평가합니다.”
**수정 범위·통과 기준:** 모델을 호출하거나 정답을 반환하지 말고 prompt 문자열을 구성합니다. 질문도 비문자열·공백뿐인 경우 거부합니다. 검증은 질문과 `직접 확인`·`unknown` 문구, field 순서, 입력 목록 보존, 잘못된 목록의 `ValueError`를 확인합니다. 예를 들어 질문을 strip한 뒤 `질문: ...`, `이미지에서 직접 확인되는 근거만 사용하세요.`, `확실하지 않으면 unknown`, `항목: ...`처럼 구성할 수 있습니다. 저장 후 `check_prompts('exercise')`의 PASS까지 확인합니다.
<!-- todo-review:requirements:end -->


In [ ]:
# 구분: 수업 핵심 셀
from tests.test_course import check_prompts
check_prompts('exercise')
print('prompt TODO PASS')


In [ ]:
# 구분: 수업 핵심 셀
# 참고 정답은 강사가 안내하는 시점에 파일 편집기에서 확인합니다.
solution_path = COURSE / 'solutions/prompts.py'
print('참고 정답 경로:', solution_path)
# 수강생 파일을 자동으로 덮어쓰지 않습니다.


### 정답 해설 — 입력부터 결과까지

**질문과 출력 조건:** 질문을 정리하고 required_fields가 list/tuple인지 확인한 뒤 항목별 공백·빈 값·중복을 검사합니다. `'label'`이라는 문자열 하나는 글자 모음으로 잘못 처리될 수 있어 거부합니다. 필드는 주어진 순서를 유지해 답변 항목을 안내합니다.

`['animal','visible_evidence']`이면 두 항목과 이미지에서 직접 확인되는 근거만 쓰라는 지시를 함께 만듭니다. `['label',' label ']`은 공백을 정리하면 중복이므로 오류입니다. 불확실하면 unknown을 요구하지만 모델이 반드시 이를 지키거나 정답을 말하는 것은 아닙니다.

**확인 질문:** 프롬프트가 깔끔해졌다는 것과 리드줄을 올바르게 찾았다는 것을 각각 어떤 근거로 평가하나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/prompts.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. `solutions/`가 연결 어댑터인 단원은 아래 실제 구현을 읽고, 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""02_image_questions.ipynb 참고 정답."""


def build_grounded_prompt(question, required_fields):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    if isinstance(required_fields, (str, bytes)) or not isinstance(required_fields, (list, tuple)):
        raise ValueError("required_fields는 문자열 list 또는 tuple이어야 합니다.")
    # 원본 순서를 보존한 새 목록으로 정리합니다. 잘못된 원소는 길이 검사로 거부합니다.
    fields = [field.strip() for field in required_fields if isinstance(field, str) and field.strip()]
    if not fields or len(fields) != len(required_fields) or len(set(fields)) != len(fields):
        raise ValueError("required_fields에는 중복 없는 비어 있지 않은 문자열이 필요합니다.")
    # 질문·시각 근거·불확실성·출력 순서를 한 문자열로 연결합니다.
    return (
        f"질문: {question.strip()}\n"
        "이미지에서 직접 확인되는 근거만 사용하세요. 보이지 않거나 확실하지 않으면 unknown이라고 쓰세요.\n"
        f"반드시 다음 항목을 이 순서로 답하세요: {', '.join(fields)}"
    )
```

</details>
<!-- todo-review:answer:end -->


In [ ]:
# 구분: 수업 핵심 셀
op = load_op('prompts')
grounded = op.build_grounded_prompt(
    '이미지에서 동물과 연결된 물체를 찾으세요.',
    ['animal', 'connected_object', 'visible_evidence'],
)
print(grounded)


> **구분: 수업 핵심**

## 3. 실제 두 요청 비교

<!-- procedure:lesson-18-a7d40f:start -->
![같은 이미지에 다른 질문 비교](../교재그림/절차/lesson-18.svg)

질문을 바꿀 때 이미지와 모델까지 함께 바꾸면 차이의 원인을 구분하기 어렵습니다.
<!-- procedure:lesson-18-a7d40f:end -->


이미지 byte와 모델 option은 같고 prompt만 다릅니다. 이것이 통제된 비교입니다. 첫 응답이 모델 적재를 포함할 수 있으므로 두 시간의 단순 대소만으로 프롬프트가 빠르다고 결론 내리지 마세요.

새 비교마다 `Data/outputs/05.VLM/comparison-<실행ID>/`를 만듭니다. `manifest.json`은 입력 해시·모델·질문·옵션을 고정하고, `prompt-comparison.jsonl`에는 **요청 하나가 끝날 때마다** 성공 또는 오류·원문·지연을 한 줄씩 추가합니다. 두 번째 요청이 실패해도 첫 성공은 파일에 남습니다.

아래 셀은 새 비교를 시작합니다. 실패하거나 중단되면 새 비교 셀을 반복하지 말고 바로 아래 **실패·미실행 요청만 재시도** 셀을 사용하세요. 이미 성공한 요청은 다시 보내지 않고 원래 오류 기록도 보존합니다. 비교 조건을 바꾸려면 새 비교를 시작합니다.


In [ ]:
# 구분: 수업 핵심 셀
client = OllamaVLMClient(timeout=300)
comparison_dir = create_prompt_comparison(
    client, IMAGES / 'dog.jpg', [('broad', broad), ('grounded', grounded)], OUTPUTS,
)
print('이번 비교 폴더:', comparison_dir)
records, destination = record_prompt_comparison(client, comparison_dir)
for record in records:
    print('\n===', record['prompt_name'], '/', record['status'], '===')
    print(record['answer'] if record['status'] == 'ok' else record['error'])
    print(json.dumps(record['timing'], ensure_ascii=False, indent=2))
print('saved:', destination)


> **구분: 수업 핵심 — 오류·중단 후 복구**

### 실패·미실행 요청만 재시도

연결·이미지·모델 상태를 복구한 뒤 아래 셀만 실행합니다. 같은 비교 폴더는 한 커널에서 차례대로 실행하세요. `status=ok` 요청은 건너뛰고 `request_error`·`interrupted`와 미실행 요청만 다시 보냅니다. 실패 시도와 재시도는 같은 JSONL에 `attempt` 번호로 모두 남습니다. 두 요청이 모두 성공했다면 새 요청 없이 기존 결과를 보여줍니다.

커널을 재시작했다면 첫 환경 셀을 실행한 뒤 `client = OllamaVLMClient(timeout=300)`과 `comparison_dir = Path('앞서 출력된 비교 폴더의 실제 경로')`를 정의하고 아래 셀을 실행합니다. 질문은 저장된 `manifest.json`에서 읽으므로 prompt TODO를 다시 실행할 필요는 없습니다. 모델과 입력 이미지가 원래 조건과 다르면 복구하거나 새 비교를 만듭니다. JSONL 자체를 쓰지 못한 오류는 터미널/셀의 예외에서 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀 — 실패·미실행 요청만 재시도
records, destination = record_prompt_comparison(client, comparison_dir)
for record in records:
    print(record['prompt_name'], record['status'], 'attempt=', record['attempt'])
    print(record['answer'] if record['status'] == 'ok' else record['error'])
print('원래 시도와 재시도 기록:', destination)


> **구분: 수업 핵심**

## 4. 응답 비교표 작성

두 응답을 읽고 아래 질문에 말로 답하세요.

| 항목 | broad | grounded |
|---|---|---|
| 실제로 보이는 동물 | 적었는가? | 적었는가? |
| 연결된 물체와 색 | 근거가 있는가? | 근거가 있는가? |
| 보이지 않는 장소·상황 추측 | 있는가? | 있는가? |
| 요청한 field 순서 | 요구하지 않음 | 지켰는가? |

이 평가는 사람이 이미지와 나란히 보고 수행합니다. 표현이 예상과 다르다는 이유만으로 실패시키지 않습니다.

### 실제 실행 사례 — 간결한 답과 정확한 답은 별도로 평가합니다

2026-09-08 배포 전 검증에서 `dog.jpg`의 넓은 질문은 `num_predict=120`의 120토큰 상한까지 생성되어 문장 중간에서 끝났습니다. 근거 중심 질문은 `animal: dog`, `connected_object: unknown`, `visible_evidence: unknown`을 출력했습니다. 원본 사진에는 보라색 리드줄이 보이므로 동물 식별은 맞았지만 연결 물체를 찾는 과제는 해결하지 못했습니다. 이는 당시 실행의 관측값이며 다음 실행의 고정 정답 출력이 아닙니다.

`num_predict`는 생성 토큰 수의 상한입니다. 토큰은 글자나 단어와 일대일 대응하지 않습니다. 출력 토큰 수가 상한에 도달하고 문장이 미완성이면 출력 잘림을 점검합니다. 원문과 `timing`의 출력 토큰 수를 함께 보세요. 응답이 짧거나 field 순서를 지켰다는 사실만으로 정답률 향상이라고 결론 내리지 않습니다.

비교표에 **동물 식별 / 연결 물체 식별 / 근거 일치 / 형식 준수 / 출력 잘림** 을 따로 기록하세요. 상한을 늘려 다시 비교한다면 새 비교 셀의 `create_prompt_comparison(..., num_predict=240)`처럼 두 질문에 같은 상한을 적용하고 모델·입력·option·원문·지연을 함께 남깁니다. 상한 증가는 문장 완성을 도울 수 있지만 인식 오류를 고친다는 보장은 없습니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

**완료 확인:** 같은 이미지에 대한 두 요청의 원문·근거 비교

[2일차 슬라이드 60쪽](https://docs.google.com/presentation/d/1mudBWptHY08YJhxmFABzzlSeSAsT1Ccc1qLA1FppqyM/edit#slide=id.d2_s055) · [배포 PDF 60쪽](../슬라이드/2일차.pdf#page=60)로 돌아가 **60–66쪽** 에서 방금 수행한 내용과 결과를 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습**

## 자습: 샘플링 option

`temperature`가 높으면 다양한 답이 나올 수 있습니다. 고정 seed와 0은 수업 비교를 안정화하지만 완전한 재현을 보장하지 않습니다. 같은 질문을 여러 번 실행할 때에는 모델 tag·digest·option도 기록해야 합니다.


In [ ]:
# 구분: 수업 후 복습·자습 셀
options = [
    {'temperature': 0, 'seed': 2026},
    {'temperature': 0.8, 'seed': 2026},
]
print(json.dumps(options, indent=2))
print('자습에서는 요청 횟수와 시간을 고려해 한 조건씩 실행하세요.')


> **구분: 수업 핵심**

## 정리

좋은 질문은 관찰 대상, 이미지 근거, 불확실성 처리와 출력 항목을 드러냅니다. 다음 노트북에서는 정지 이미지 한 장이 아니라 카메라가 빠르게 만드는 프레임 중 **언제 어떤 한 장을 보낼지** 결정합니다.
